<a href="https://colab.research.google.com/github/akgrenier/cloudlearn-companion/blob/main/notebooks/bonus-unit1/bonus-unit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bonus Unit 1: Let's train Huggy the Dog 🐶 to fetch a stick

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit2/thumbnail.png" alt="Bonus Unit 1Thumbnail">

In this notebook, we'll reinforce what we learned in the first Unit by **teaching Huggy the Dog to fetch the stick and then play with it directly in your browser**

⬇️ Here is an example of what **you will achieve at the end of the unit.** ⬇️ (launch ▶ to see)

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.mp4" type="video/mp4"></video>

### The environment 🎮

- Huggy the Dog, an environment created by [Thomas Simonini](https://twitter.com/ThomasSimonini) based on [Puppo The Corgi](https://blog.unity.com/technology/puppo-the-corgi-cuteness-overload-with-the-unity-ml-agents-toolkit)

### The library used 📚

- [MLAgents](https://github.com/Unity-Technologies/ml-agents)

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the Github Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Understand **the state space, action space and reward function used to train Huggy**.
- **Train your own Huggy** to fetch the stick.
- Be able to play **with your trained Huggy directly in your browser**.




## This notebook is from Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

And more check 📚 the syllabus 👉 https://simoninithomas.github.io/deep-rl-course

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️

Before diving into the notebook, you need to:

🔲 📚 **Develop an understanding of the foundations of Reinforcement learning** (MC, TD, Rewards hypothesis...) by doing Unit 1

🔲 📚 **Read the introduction to Huggy** by doing Bonus Unit 1

## Set the GPU 💪
- To **accelerate the agent's training, we'll use a GPU**. To do that, go to `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## Clone the repository 🔽

- We need to clone the repository, that contains **ML-Agents.**

In [ ]:
# Ensure we are in /content before cloning
%cd /content

# Clean up previous clone attempts if any
!/usr/bin/rm -rf ml-agents

# Clone the repository (can take 3min) - Removed %%capture to see output
print("Cloning ml-agents repository:")
!/usr/bin/git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

# Verify ml-agents directory exists after cloning
print("Contents of /content after cloning ml-agents:")
!/bin/ls -F /content

## Setup the Virtual Environment 🔽
- In order for the **ML-Agents** to run successfully in Colab,  Colab's Python version must meet the library's Python requirements.

- We can check for the supported Python version under the `python_requires` parameter in the `setup.py` files. These files are required to set up the **ML-Agents** library for use and can be found in the following locations:
  - `/content/ml-agents/ml-agents/setup.py`
  - `/content/ml-agents/ml-agents-envs/setup.py`

- Colab's Current Python version(can be checked using `!python --version`) doesn't match the library's `python_requires` parameter, as a result installation may silently fail and lead to errors like these, when executing the same commands later:
  - `/bin/bash: line 1: mlagents-learn: command not found`
  - `/bin/bash: line 1: mlagents-push-to-hf: command not found`

- To resolve this, we'll create a virtual environment with a Python version compatible with the **ML-Agents** library.

`Note:` *For future compatibility, always check the `python_requires` parameter in the installation files and set your virtual environment to the maximum supported Python version in the given below script if the Colab's Python version is not compatible*

In [ ]:
# Colab's Current Python Version (Incompatible with ML-Agents)
!python --version

In [ ]:
# Reset and set a standard PATH for robustness in case it's corrupted
%env PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin

# Update package lists
!/usr/bin/apt-get update -y

# Install gnupg and debianutils (which provides 'which' command)
!/usr/bin/apt-get install -y gnupg debianutils

# Verify 'which' command immediately after installation
print("Verifying 'which' command immediately after installing debianutils:")
!/usr/bin/which which

# Reinstall Python 3.10 and venv module to ensure clean symlinks and setup
!/usr/bin/apt-get install --reinstall -y python3.10 python3.10-dev python3.10-venv

# Remove any existing venv directory to ensure a clean start
!rm -rf myenv

# Verify what /usr/bin/python3.10 points to before creating venv
print("Verifying /usr/bin/python3.10 symlink:")
!ls -l /usr/bin/python3.10
print("Verifying /usr/bin/python3.10 version:")
!/usr/bin/python3.10 --version

# Create a virtual environment named 'myenv' using the absolute path to python3.10
!/usr/bin/python3.10 -m venv myenv

# --- FIX: Explicitly correct virtual environment symlinks to point to Python 3.10 ---
# Remove incorrect symlinks that might be pointing to the system's default Python (3.12)
!rm -f /content/myenv/bin/python /content/myenv/bin/python3 /content/myenv/bin/python3.12

# Create correct symlinks to Python 3.10 executable
!ln -s /usr/bin/python3.10 /content/myenv/bin/python
!ln -s /usr/bin/python3.10 /content/myenv/bin/python3
# Note: mlagents-learn might specifically look for python3.10, so a direct symlink is good practice.
!ln -s /usr/bin/python3.10 /content/myenv/bin/python3.10
# ----------------------------------------------------------------------------------

# IMPORTANT: Inspect the contents of the virtual environment's bin directory to see where symlinks point
print("Inspecting /content/myenv/bin directory (after symlink fix):")
!ls -l /content/myenv/bin

# Verify pyvenv.cfg content
print("Verifying myenv/pyvenv.cfg content:")
!cat myenv/pyvenv.cfg

# Explicitly set environment variables for the virtual environment
# Prepend /content/myenv/bin to PATH to ensure it's prioritized for current session
%env PATH=/content/myenv/bin:$PATH
%env VIRTUAL_ENV=/content/myenv

print("Python 3.10 virtual environment 'myenv' set up.")
print("Verifying Python version within the virtual environment (after symlink fix):")
!/content/myenv/bin/python --version
!/content/myenv/bin/pip --version

In [ ]:
# Verify the active Python and pip versions after virtual environment setup
print("Currently active Python:")
# Use the known working 'which' to find 'python' in the PATH
!/usr/bin/which python
!python --version

print("Currently active Pip:")
# Use the known working 'which' to find 'pip' in the PATH
!/usr/bin/which pip
!pip --version

print("Verifying Python and Pip directly from the virtual environment's bin directory:")
!/content/myenv/bin/python --version
!/content/myenv/bin/pip --version

## Installing the dependencies 🔽

In [ ]:
# First, let's see what's in /content to locate the ml-agents directory
print("Contents of /content before attempting to change directory:")
!/bin/ls -F /content

# Go inside the repository and install the package
# Assuming ml-agents was cloned directly into /content, otherwise adjust path
%cd /content/ml-agents
!/content/myenv/bin/pip install -e ./ml-agents-envs
!/content/myenv/bin/pip install -e ./ml-agents

print("Verifying mlagents-learn after installation:")
!/bin/ls -l /content/myenv/bin/mlagents-learn

## Download and move the environment zip file in `./trained-envs-executables/linux/`

- Our environment executable is in a zip file.
- We need to download it and place it to `./trained-envs-executables/linux/`

In [ ]:
# Create the necessary directory structure using absolute paths and -p for robustness
!/bin/mkdir -p /content/trained-envs-executables/linux

# Verify the creation of the directory
print("Verifying creation of /content/trained-envs-executables/linux/")
!/bin/ls -ld /content/trained-envs-executables/linux/

We downloaded the file Huggy.zip from https://github.com/huggingface/Huggy using `wget`

In [ ]:
# Ensure we are in /content before downloading
%cd /content

target_zip_path = "/content/trained-envs-executables/linux/Huggy.zip"

# Verify target directory exists before attempting to download
print(f"Verifying target directory existence: {os.path.dirname(target_zip_path)}")
!/bin/ls -ld {os.path.dirname(target_zip_path)}

print(f"Downloading Huggy.zip to: {target_zip_path}")

# Download to the specific absolute directory
!/usr/bin/wget "https://github.com/huggingface/Huggy/raw/main/Huggy.zip" -O "{target_zip_path}"

# Verify that the Huggy.zip file is present after download
print("Verifying Huggy.zip after download:")
!/bin/ls -l "{target_zip_path}"

In [ ]:
# Ensure we are in /content before anything else
%cd /content

executable_dir = "/content/trained-envs-executables/linux"
executable_path = f"{executable_dir}/Huggy/Huggy.x86_64"
zip_path = f"{executable_dir}/Huggy.zip"

# Remove any existing Huggy directory to ensure a clean unzip
print(f"Removing existing Huggy directory at {executable_dir}/Huggy")
!/bin/rm -rf "{executable_dir}/Huggy"

# Unzip Huggy.zip into the executable_dir
print(f"Unzipping {zip_path} into {executable_dir}/")
# Using -d to specify destination directory to ensure it extracts correctly
!/usr/bin/unzip -o "{zip_path}" -d "{executable_dir}"

# Verify the contents of the directory after unzipping
print(f"Contents of {executable_dir}/Huggy/ after unzipping:")
!/bin/ls -l "{executable_dir}/Huggy/"

# Verify the executable exists and apply permissions
import os
if os.path.exists(executable_path):
    print(f"Executable found at {executable_path}. Applying permissions.")
    !/bin/chmod 755 "{executable_path}"
    print("Permissions applied. Verifying new permissions:")
    !/bin/ls -l "{executable_path}"
else:
    print(f"Error: Executable not found at the specified path: {executable_path}")
    print("This is a critical error. Please inspect the unzip output carefully.")

Make sure your file is accessible

In [ ]:
executable_path = "/content/trained-envs-executables/linux/Huggy/Huggy.x86_64"

print(f"Final check for executable at: {executable_path}")
!/bin/ls -l {executable_path}

## Let's recap how this environment works

### The State Space: what Huggy "perceives."

Huggy doesn't "see" his environment. Instead, we provide him information about the environment:

- The target (stick) position
- The relative position between himself and the target
- The orientation of his legs.

Given all this information, Huggy **can decide which action to take next to fulfill his goal**.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.jpg" alt="Huggy" width="100%">


### The Action Space: what moves Huggy can do
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-action.jpg" alt="Huggy action" width="100%">

**Joint motors drive huggy legs**. It means that to get the target, Huggy needs to **learn to rotate the joint motors of each of his legs correctly so he can move**.

### The Reward Function

The reward function is designed so that **Huggy will fulfill his goal** : fetch the stick.

Remember that one of the foundations of Reinforcement Learning is the *reward hypothesis*: a goal can be described as the **maximization of the expected cumulative reward**.

Here, our goal is that Huggy **goes towards the stick but without spinning too much**. Hence, our reward function must translate this goal.

Our reward function:

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/reward.jpg" alt="Huggy reward function" width="100%">

- *Orientation bonus*: we **reward him for getting close to the target**.
- *Time penalty*: a fixed-time penalty given at every action to **force him to get to the stick as fast as possible**.
- *Rotation penalty*: we penalize Huggy if **he spins too much and turns too quickly**.
- *Getting to the target reward*: we reward Huggy for **reaching the target**.

## Create the Huggy config file

- In ML-Agents, you define the **training hyperparameters into config.yaml files.**

- For the scope of this notebook, we're not going to modify the hyperparameters, but if you want to try as an experiment, you should also try to modify some other hyperparameters, Unity provides very [good documentation explaining each of them here](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md).

- But we need to create a config file for Huggy.

  - To do that click on Folder logo on the left of your screen.

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create_file.png" alt="Create file" width="10%">

  - Go to `/content/ml-agents/config/ppo`
  - Right mouse click and create a new file called `Huggy.yaml`

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create-huggy.png" alt="Create huggy.yaml" width="20%">

- Copy and paste the content below 🔽

In [ ]:
The following content should be saved as `/content/ml-agents/config/ppo/Huggy.yaml`:
```yaml
behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000
```

In [ ]:
# Create the config directory if it doesn't exist
!/bin/mkdir -p /content/ml-agents/config/ppo

In [ ]:
%%writefile /content/ml-agents/config/ppo/Huggy.yaml
behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000

- Don't forget to save the file!

- **In the case you want to modify the hyperparameters**, in Google Colab notebook, you can click here to open the config.yaml: `/content/ml-agents/config/ppo/Huggy.yaml`

- For instance **if you want to save more models during the training** (for now, we save every 200,000 training timesteps). You need to modify:
  - `checkpoint_interval`: The number of training timesteps collected between each checkpoint.
  - `keep_checkpoints`: The maximum number of model checkpoints to keep.

=> Just keep in mind that **decreasing the `checkpoint_interval` means more models to upload to the Hub and so a longer uploading time**
We’re now ready to train our agent 🔥.

## Train our agent

To train our agent, we just need to **launch mlagents-learn and select the executable containing the environment.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mllearn.png" alt="ml learn function" width="100%">

With ML Agents, we run a training script. We define four parameters:

1. `mlagents-learn <config>`: the path where the hyperparameter config file is.
2. `--env`: where the environment executable is.
3. `--run-id`: the name you want to give to your training run id.
4. `--no-graphics`: to not launch the visualization during the training.

Train the model and use the `--resume` flag to continue training in case of interruption.

> It will fail first time when you use `--resume`, try running the block again to bypass the error.



The training will take 30 to 45min depending on your machine (don't forget to **set up a GPU**), go take a ☕️you deserve it 🤗.

In [ ]:
!/content/myenv/bin/mlagents-learn /content/ml-agents/config/ppo/Huggy.yaml --env=/content/trained-envs-executables/linux/Huggy/Huggy.x86_64 --run-id="Huggy2" --no-graphics --force

## Push the agent to the 🤗 Hub

- Now that we trained our agent, we’re **ready to push it to the Hub to be able to play with Huggy on your browser🔥.**

To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account to HF ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and paste the token

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

If you don't want to use a Google Colab or a Jupyter Notebook, you need to use this command instead: `huggingface-cli login`

Then, we simply need to run `mlagents-push-to-hf`.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mlpush.png" alt="ml learn function" width="100%">

And we define 4 parameters:

1. `--run-id`: the name of the training run id.
2. `--local-dir`: where the agent was saved, it’s results/<run_id name>, so in my case results/First Training.
3. `--repo-id`: the name of the Hugging Face repo you want to create or update. It’s always <your huggingface username>/<the repo name>
If the repo does not exist **it will be created automatically**
4. `--commit-message`: since HF repos are git repository you need to define a commit message.

In [ ]:
!mlagents-push-to-hf --run-id="HuggyTraining" --local-dir="./results/Huggy2" --repo-id="ThomasSimonini/ppo-Huggy" --commit-message="Huggy"

Else, if everything worked you should have this at the end of the process(but with a different url 😆) :



```
Your model is pushed to the hub. You can view your model here: https://huggingface.co/ThomasSimonini/ppo-Huggy
```

It’s the link to your model repository. The repository contains a model card that explains how to use the model, your Tensorboard logs and your config file. **What’s awesome is that it’s a git repository, which means you can have different commits, update your repository with a new push, open Pull Requests, etc.**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/modelcard.png" alt="ml learn function" width="100%">

But now comes the best: **being able to play with Huggy online 👀.**

## Play with your Huggy 🐕

This step is the simplest:

- Open the game Huggy in your browser: https://huggingface.co/spaces/ThomasSimonini/Huggy

- Click on Play with my Huggy model

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/load-huggy.jpg" alt="load-huggy" width="100%">

1. In step 1, type your username (your username is case sensitive: for instance, my username is ThomasSimonini not thomassimonini or ThOmasImoNInI) and click on the search button.

2. In step 2, select your model repository.

3. In step 3, **choose which model you want to replay**:
  - I have multiple ones, since we saved a model every 500000 timesteps.
  - But since I want the more recent, I choose `Huggy.onnx`

👉 What’s nice **is to try with different models steps to see the improvement of the agent.**

Congrats on finishing this bonus unit!

You can now sit and enjoy playing with your Huggy 🐶. And don't **forget to spread the love by sharing Huggy with your friends 🤗**. And if you share about it on social media, **please tag us @huggingface and me @simoninithomas**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-cover.jpeg" alt="Huggy cover" width="100%">


## Keep Learning, Stay  awesome 🤗